In [1]:
!pip install fastapi uvicorn nest_asyncio "pydantic<3" requests


In [2]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, Literal, List
import datetime as dt

app = FastAPI(
    title="Library AI Assistant",
    description=(
        "Backend for a Library AI Assistant with implicit vs explicit modes.\n"
        "Implicit = limited knowledge (public catalog only)\n"
        "Explicit = full knowledge (catalog + borrowing / inventory details)"
    ),
)

# -----------------------------
# In-memory "database"
# -----------------------------
BOOKS = [
    {
        "id": 1,
        "title": "Introduction to Data Science",
        "author": "Smith, A.",
        "genre": "Education",
        "total_copies": 5,
        "available_copies": 2,
    },
    {
        "id": 2,
        "title": "Machine Learning Basics",
        "author": "Lee, B.",
        "genre": "Technology",
        "total_copies": 3,
        "available_copies": 1,
    },
    {
        "id": 3,
        "title": "Information Systems in Practice",
        "author": "Taylor, C.",
        "genre": "Information Systems",
        "total_copies": 4,
        "available_copies": 4,
    },
]

BORROW_RECORDS = [
    {
        "user_id": "u1001",
        "book_id": 1,
        "borrowed_on": "2025-11-15",
        "due_date": "2025-12-01",
    },
    {
        "user_id": "u1002",
        "book_id": 2,
        "borrowed_on": "2025-11-20",
        "due_date": "2025-12-05",
    },
]

# -----------------------------
# Prompt-style system messages (for your PPT)
# -----------------------------
IMPLICIT_SYSTEM_PROMPT = """
You are a LIBRARY ASSISTANT in IMPLICIT mode.
You are only allowed to use PUBLIC CATALOG information:
- Book titles
- Authors
- Genres
- Whether the library has this book or not (yes/no)

You MUST NOT:
- Show how many copies are available
- Show who borrowed which book
- Show due dates or user-level data

If the user asks for private data, explain that implicit mode
cannot access detailed inventory or user records.
"""

EXPLICIT_SYSTEM_PROMPT = """
You are a LIBRARY ASSISTANT in EXPLICIT mode.
The user has granted explicit consent.

You can use:
- Full catalog information
- Inventory counts (total and available copies)
- Borrowing records (who borrowed what, due dates)

You must still respect privacy in your explanations,
but you are allowed to answer questions about inventory
and borrowing history when asked directly by the authorized user.
"""

# -----------------------------
# Request / Response models
# -----------------------------
class AskRequest(BaseModel):
    mode: Literal["implicit", "explicit"]
    question: str
    user_id: Optional[str] = None
    consent: bool = False  # must be True for explicit access


class AskResponse(BaseModel):
    mode: str
    answer: str
    used_private_data: bool
    matched_books: Optional[List[dict]] = None


# -----------------------------
# Helper functions
# -----------------------------
def find_books_by_title_fragment(fragment: str) -> List[dict]:
    fragment = fragment.lower()
    return [b for b in BOOKS if fragment in b["title"].lower()]


def format_book_public(book: dict) -> str:
    return f"{book['title']} (by {book['author']}, genre: {book['genre']})"


def format_book_private(book: dict) -> str:
    return (
        f"{book['title']} (by {book['author']}, genre: {book['genre']}) "
        f"- total copies: {book['total_copies']}, "
        f"available: {book['available_copies']}"
    )


def get_borrowers_for_book(book_id: int) -> List[dict]:
    return [r for r in BORROW_RECORDS if r["book_id"] == book_id]


# -----------------------------
# "AI logic" – implicit vs explicit
# -----------------------------
def answer_implicit(question: str) -> AskResponse:
    q = question.lower().strip()

    # List books (public only)
    if "list" in q and "book" in q:
        books_str = "; ".join(format_book_public(b) for b in BOOKS)
        ans = f"Here are some books in our catalog: {books_str}."
        return AskResponse(
            mode="implicit",
            answer=ans,
            used_private_data=False,
            matched_books=BOOKS,
        )

    # Check if we have a specific book
    if "have" in q and "book" in q:
        fragment = q.split("book")[-1].strip(" ?.")
        if not fragment:
            fragment = q
        matches = find_books_by_title_fragment(fragment)
        if matches:
            titles = "; ".join(format_book_public(b) for b in matches)
            ans = (
                "Yes, we have the following book(s) in our catalog (implicit mode): "
                f"{titles}. I cannot show detailed inventory in this mode."
            )
        else:
            ans = "I couldn't find that title in the public catalog."
        return AskResponse(
            mode="implicit",
            answer=ans,
            used_private_data=False,
            matched_books=matches or None,
        )

    # Ask for copies / inventory -> blocked in implicit
    if "how many" in q or "copies left" in q:
        ans = (
            "You are in IMPLICIT mode. I can confirm if a book exists, "
            "but I cannot show how many copies are left or detailed inventory. "
            "Please switch to EXPLICIT mode with consent if you need that."
        )
        return AskResponse(
            mode="implicit",
            answer=ans,
            used_private_data=False,
        )

    # Fallback
    ans = (
        "Implicit mode can only use public catalog information. "
        "You can ask things like: 'List the books', "
        "or 'Do you have Machine Learning Basics?'."
    )
    return AskResponse(mode="implicit", answer=ans, used_private_data=False)


def answer_explicit(question: str, user_id: Optional[str]) -> AskResponse:
    q = question.lower().strip()
    used_private = False

    # Inventory: how many copies left?
    if "how many" in q or "copies left" in q:
        fragment = ""
        if "of" in q:
            fragment = q.split("of")[-1].strip(" ?.")
        if not fragment:
            fragment = q

        matches = find_books_by_title_fragment(fragment)
        if not matches:
            ans = "I could not find that book in the catalog."
            return AskResponse(
                mode="explicit",
                answer=ans,
                used_private_data=False,
            )

        used_private = True
        descriptions = "; ".join(format_book_private(b) for b in matches)
        ans = (
            "Here is the detailed inventory information (explicit mode): "
            f"{descriptions}."
        )
        return AskResponse(
            mode="explicit",
            answer=ans,
            used_private_data=used_private,
            matched_books=matches,
        )

    # Who borrowed / due date
    if "who borrowed" in q or "who has" in q or "due date" in q:
        fragment = ""
        if "of" in q:
            fragment = q.split("of")[-1].strip(" ?.")
        if not fragment:
            fragment = q

        matches = find_books_by_title_fragment(fragment)
        if not matches:
            ans = "I could not find that book in the catalog."
            return AskResponse(
                mode="explicit",
                answer=ans,
                used_private_data=False,
            )

        used_private = True
        pieces = []
        for b in matches:
            borrowers = get_borrowers_for_book(b["id"])
            if not borrowers:
                pieces.append(
                    f"'{b['title']}' is currently not borrowed by anyone."
                )
            else:
                for rec in borrowers:
                    pieces.append(
                        f"Book '{b['title']}' is borrowed by user {rec['user_id']} "
                        f"and is due on {rec['due_date']}."
                    )

        ans = " ".join(pieces)
        return AskResponse(
            mode="explicit",
            answer=ans,
            used_private_data=used_private,
            matched_books=matches,
        )

    # Full list with inventory
    if "list" in q and "book" in q:
        used_private = True
        books_str = "; ".join(format_book_private(b) for b in BOOKS)
        ans = f"Here are the books with full inventory: {books_str}."
        return AskResponse(
            mode="explicit",
            answer=ans,
            used_private_data=used_private,
            matched_books=BOOKS,
        )

    # Fallback
    ans = (
        "Explicit mode allows detailed answers about inventory and borrowing records. "
        "You can ask: 'How many copies of Machine Learning Basics are left?' "
        "or 'Who borrowed Introduction to Data Science?'."
    )
    return AskResponse(mode="explicit", answer=ans, used_private_data=False)


# -----------------------------
# FastAPI endpoints
# -----------------------------
@app.post("/ask", response_model=AskResponse)
def ask_library_ai(payload: AskRequest):
    """
    Main endpoint your client/Colab code calls.

    - mode = 'implicit'  -> limited public info only
    - mode = 'explicit'  -> full access, requires consent=True
    """
    if payload.mode == "explicit" and not payload.consent:
        raise HTTPException(
            status_code=403,
            detail=(
                "Explicit mode requested but consent=False. "
                "Set consent=True when the user has granted explicit permission."
            ),
        )

    if payload.mode == "implicit":
        return answer_implicit(payload.question)
    else:
        return answer_explicit(payload.question, payload.user_id)


@app.get("/prompts")
def get_prompts():
    """
    Helper endpoint to show the implicit/explicit system prompts.
    Useful for screenshots in your PPT.
    """
    return {
        "implicit_system_prompt": IMPLICIT_SYSTEM_PROMPT.strip(),
        "explicit_system_prompt": EXPLICIT_SYSTEM_PROMPT.strip(),
    }


@app.get("/")
def root():
    return {
        "message": "Library AI Assistant backend is running.",
        "time": dt.datetime.now().isoformat(),
    }


In [3]:
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def run_server():
    # 'app' is the FastAPI instance from the previous cell
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait a moment for the server to boot
time.sleep(3)
print("✅ FastAPI server started on http://127.0.0.1:8000")


INFO:     Started server process [172]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ FastAPI server started on http://127.0.0.1:8000


In [4]:
import requests

BASE_URL = "http://127.0.0.1:8000"

# ---- Test implicit mode ----
resp_implicit = requests.post(
    f"{BASE_URL}/ask",
    json={
        "mode": "implicit",
        "question": "Do you have Machine Learning Basics book?",
        "user_id": "u1001",
        "consent": False
    },
)
print("IMPLICIT RESPONSE:")
print(resp_implicit.json())

# ---- Test explicit mode ----
resp_explicit = requests.post(
    f"{BASE_URL}/ask",
    json={
        "mode": "explicit",
        "question": "How many copies of Machine Learning Basics are left?",
        "user_id": "u1001",
        "consent": True
    },
)
print("\nEXPLICIT RESPONSE:")
print(resp_explicit.json())

# ---- Show prompts for your PPT ----
resp_prompts = requests.get(f"{BASE_URL}/prompts")
print("\nPROMPTS ENDPOINT:")
print(resp_prompts.json())


INFO:     127.0.0.1:50568 - "POST /ask HTTP/1.1" 200 OK
IMPLICIT RESPONSE:
{'mode': 'implicit', 'answer': "I couldn't find that title in the public catalog.", 'used_private_data': False, 'matched_books': None}
INFO:     127.0.0.1:50576 - "POST /ask HTTP/1.1" 200 OK

EXPLICIT RESPONSE:
{'mode': 'explicit', 'answer': 'I could not find that book in the catalog.', 'used_private_data': False, 'matched_books': None}
INFO:     127.0.0.1:50586 - "GET /prompts HTTP/1.1" 200 OK

PROMPTS ENDPOINT:
{'implicit_system_prompt': 'You are a LIBRARY ASSISTANT in IMPLICIT mode.\nYou are only allowed to use PUBLIC CATALOG information:\n- Book titles\n- Authors\n- Genres\n- Whether the library has this book or not (yes/no)\n\nYou MUST NOT:\n- Show how many copies are available\n- Show who borrowed which book\n- Show due dates or user-level data\n\nIf the user asks for private data, explain that implicit mode\ncannot access detailed inventory or user records.', 'explicit_system_prompt': 'You are a LIBRARY A

In [5]:
!npm install -g localtunnel


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 5s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇npm notice
npm notice New major version of npm available! 10.8.2 -> 11.6.4
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.6.4
npm notice To update run: npm install -g npm@11.6.4
npm notice
⠇

In [6]:
!pip install fastapi uvicorn nest_asyncio "pydantic<3"


In [7]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("🚀 FastAPI started on port 8000 (internal).")


🚀 FastAPI started on port 8000 (internal).


In [8]:
import subprocess
import threading
import time

def start_localtunnel():
    print("🌐 Starting public tunnel...")
    # This command exposes port 8000 on a public URL
    proc = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)

    for line in proc.stdout:
        decoded = line.decode()
        if "your url is:" in decoded.lower():
            public_url = decoded.split("is:")[1].strip()
            print("🔗 PUBLIC URL:", public_url)
            break

thread = threading.Thread(target=start_localtunnel, daemon=True)
thread.start()

time.sleep(5)  # wait for the tunnel to start
print("Tunnel should be ready above.")


🌐 Starting public tunnel...
Tunnel should be ready above.


In [10]:
# Kill any old servers or tunnels
!pkill -f uvicorn || true
!pkill -f lt || true

print("✅ Killed any existing uvicorn/localtunnel processes.")


^C
^C
✅ Killed any existing uvicorn/localtunnel processes.


In [11]:
import subprocess
import threading
import time
import re

def start_localtunnel():
    print("🌐 Starting public tunnel...")
    cmd = ["lt", "--port", "8000"]

    # Start lt process
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # Read stdout line by line
    while True:
        line = process.stdout.readline()
        if not line:
            break

        print(line.strip())  # show all output for debugging

        # Detect URL
        match = re.search(r"(https?://[a-zA-Z0-9.-]+\.loca\.lt)", line)
        if match:
            public_url = match.group(1)
            print("\n🔗 PUBLIC URL FOUND:", public_url)
            print("🚀 Your API is now live!")
            return public_url

    print("❌ ERROR: No URL detected.")

# Start in background
thread = threading.Thread(target=start_localtunnel, daemon=True)
thread.start()

# Give tunnel time to start
time.sleep(5)
print("⏳ Waiting for URL... look above for the printed link.\n")


🌐 Starting public tunnel...
⏳ Waiting for URL... look above for the printed link.



In [12]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("🚀 FastAPI started on port 8000 inside Colab.")


🚀 FastAPI started on port 8000 inside Colab.


INFO:     Started server process [172]
INFO:     Waiting for application startup.


In [15]:
import subprocess
import re

print("Starting LocalTunnel...")

process = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

while True:
    line = process.stdout.readline()
    if not line:
        break

    print(line.strip())

    # Extract URL
    url_match = re.search(r"(https:\/\/[a-zA-Z0-9.-]+\.loca\.lt)", line)
    if url_match:
        print("\n🔗 Tunnel URL:", url_match.group(1))

    # Extract password/access token
    token_match = re.search(r"access token:\s*([a-zA-Z0-9_-]+)", line)
    if token_match:
        print("\n🔑 Access Token:", token_match.group(1))
        print("Use this token to open the URL in your browser.")


Starting LocalTunnel...
your url is: https://tangy-nails-create.loca.lt

🔗 Tunnel URL: https://tangy-nails-create.loca.lt
/tools/node/lib/node_modules/localtunnel/bin/lt.js:81
throw err;
^

Error: connection refused: localtunnel.me:16679 (check your firewall settings)
at Socket.<anonymous> (/tools/node/lib/node_modules/localtunnel/lib/TunnelCluster.js:52:11)
at Socket.emit (node:events:524:28)
at emitErrorNT (node:internal/streams/destroy:169:8)
at emitErrorCloseNT (node:internal/streams/destroy:128:3)
at process.processTicksAndRejections (node:internal/process/task_queues:82:21)

Node.js v20.19.0


In [13]:
!lt --port 8000


your url is: https://tangy-cloths-work.loca.lt

🔗 PUBLIC URL FOUND: https://tangy-cloths-work.loca.lt
🚀 Your API is now live!
^C


In [16]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared-linux-amd64.deb


Selecting previously unselected package cloudflared.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2025.11.1) ...
Setting up cloudflared (2025.11.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [17]:
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)
print("✅ FastAPI server started on http://127.0.0.1:8000")


INFO:     Started server process [172]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


✅ FastAPI server started on http://127.0.0.1:8000


In [18]:
!cloudflared tunnel --url http://localhost:8000 --no-autoupdate


2025-12-05T05:38:31Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-12-05T05:38:31Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-12-05T05:38:34Z INF +--------------------------------------------------------------------------------------------+
2025-12-05T05:38:34Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2025-12-05T05:38:34Z INF |  https://ladder-establish-expand-different.trycloudfla

In [19]:
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)
print("✅ FastAPI server started on http://127.0.0.1:8000")


INFO:     Started server process [172]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


✅ FastAPI server started on http://127.0.0.1:8000


In [20]:
import requests

BASE_URL = "http://127.0.0.1:8000"

# IMPLICIT
r1 = requests.post(
    f"{BASE_URL}/ask",
    json={
        "mode": "implicit",
        "question": "Do you have Machine Learning Basics book?",
        "user_id": "u1001",
        "consent": False
    }
)
print("IMPLICIT:")
print(r1.json())

# EXPLICIT
r2 = requests.post(
    f"{BASE_URL}/ask",
    json={
        "mode": "explicit",
        "question": "How many copies of Machine Learning Basics are left?",
        "user_id": "u1001",
        "consent": True
    }
)
print("\nEXPLICIT:")
print(r2.json())

# PROMPTS
r3 = requests.get(f"{BASE_URL}/prompts")
print("\nPROMPTS:")
print(r3.json())


ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-21' coro=<Server.serve() done, defined at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:69> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 164, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1584, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-1723066382.py", li

INFO:     127.0.0.1:38484 - "POST /ask HTTP/1.1" 200 OK
IMPLICIT:
{'mode': 'implicit', 'answer': "I couldn't find that title in the public catalog.", 'used_private_data': False, 'matched_books': None}
INFO:     127.0.0.1:38498 - "POST /ask HTTP/1.1" 200 OK

EXPLICIT:
{'mode': 'explicit', 'answer': 'I could not find that book in the catalog.', 'used_private_data': False, 'matched_books': None}
INFO:     127.0.0.1:38514 - "GET /prompts HTTP/1.1" 200 OK

PROMPTS:
{'implicit_system_prompt': 'You are a LIBRARY ASSISTANT in IMPLICIT mode.\nYou are only allowed to use PUBLIC CATALOG information:\n- Book titles\n- Authors\n- Genres\n- Whether the library has this book or not (yes/no)\n\nYou MUST NOT:\n- Show how many copies are available\n- Show who borrowed which book\n- Show due dates or user-level data\n\nIf the user asks for private data, explain that implicit mode\ncannot access detailed inventory or user records.', 'explicit_system_prompt': 'You are a LIBRARY ASSISTANT in EXPLICIT mode.\

In [21]:
import requests

BASE_URL = "http://127.0.0.1:8000"

print("🔍 Testing Library AI Assistant...\n")

# -------------------------------
# 1️⃣ Test IMPLICIT mode
# -------------------------------
implicit_payload = {
    "mode": "implicit",
    "question": "Do you have the book Machine Learning Basics?",
    "user_id": "u1001",
    "consent": False
}

r_implicit = requests.post(f"{BASE_URL}/ask", json=implicit_payload)

print("=== IMPLICIT MODE RESPONSE ===")
print(r_implicit.json())
print("\n")


# -------------------------------
# 2️⃣ Test EXPLICIT mode
# -------------------------------
explicit_payload = {
    "mode": "explicit",
    "question": "How many copies of Machine Learning Basics are left?",
    "user_id": "u1001",
    "consent": True
}

r_explicit = requests.post(f"{BASE_URL}/ask", json=explicit_payload)

print("=== EXPLICIT MODE RESPONSE ===")
print(r_explicit.json())
print("\n")


# -------------------------------
# 3️⃣ GET system prompts
# -------------------------------
r_prompts = requests.get(f"{BASE_URL}/prompts")

print("=== SYSTEM PROMPTS ===")
print(r_prompts.json())
print("\n")

print("✔ All tests completed successfully!")


🔍 Testing Library AI Assistant...

INFO:     127.0.0.1:37054 - "POST /ask HTTP/1.1" 200 OK
=== IMPLICIT MODE RESPONSE ===
{'mode': 'implicit', 'answer': 'Yes, we have the following book(s) in our catalog (implicit mode): Machine Learning Basics (by Lee, B., genre: Technology). I cannot show detailed inventory in this mode.', 'used_private_data': False, 'matched_books': [{'id': 2, 'title': 'Machine Learning Basics', 'author': 'Lee, B.', 'genre': 'Technology', 'total_copies': 3, 'available_copies': 1}]}


INFO:     127.0.0.1:37060 - "POST /ask HTTP/1.1" 200 OK
=== EXPLICIT MODE RESPONSE ===
{'mode': 'explicit', 'answer': 'I could not find that book in the catalog.', 'used_private_data': False, 'matched_books': None}


INFO:     127.0.0.1:37064 - "GET /prompts HTTP/1.1" 200 OK
=== SYSTEM PROMPTS ===
{'implicit_system_prompt': 'You are a LIBRARY ASSISTANT in IMPLICIT mode.\nYou are only allowed to use PUBLIC CATALOG information:\n- Book titles\n- Authors\n- Genres\n- Whether the library ha